In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc gem-suite matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Tranziția de fază Nishimori

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Estimare de utilizare: 3 minute pe un procesor Heron r2 (NOTĂ: Aceasta este doar o estimare. Timpul tău de execuție poate varia.)*
## Rezultate de învățare
După parcurgerea acestui tutorial, utilizatorii pot aștepta următoarele rezultate:
- Înțelegerea tranziției de fază Nishimori și modul în care aceasta se manifestă ca apariție a imbricării pe distanță lungă în modelul Ising cu legături aleatoare.
- Implementarea protocolului *generarea imbricării prin măsurătoare* (GEM) pe hardware cuantic folosind măsurători în mijlocul circuitului și circuite de adâncime constantă.
- Caracterizarea tranziției prin extragerea corelației pe două puncte și a varianței normalizate a magnetizării din datele experimentale.

## Cerințe preliminare
Recomandăm familiarizarea cu următoarele subiecte înainte de a parcurge acest tutorial:
- Ghidul [Măsurarea qubiților](/guides/measure-qubits), în special secțiunea despre măsurătorile în mijlocul circuitului pe care se bazează protocolul GEM.
- [Simulare exactă și zgomotoasă cu primitivele Qiskit Aer](/guides/simulate-with-qiskit-aer), care este modalitatea prin care este executată secțiunea la scară mică.
- [Imbricare pe distanță lungă cu circuite dinamice](/tutorials/long-range-entanglement), un tutorial companion care folosește același paradigmă de imbricare bazată pe măsurătoare.
- [Rețeaua hex grea](https://www.ibm.com/quantum/blog/heavy-hex-lattice), topologia hardware IBM&reg; pe care este construită rețeaua de plachete.
## Context
Acest tutorial demonstrează cum se realizează o tranziție de fază Nishimori pe un procesor cuantic. Experimentul a fost descris inițial în [*Realizing the Nishimori transition across the error threshold for constant-depth quantum circuits*](https://arxiv.org/abs/2309.02863).

Tranziția de fază Nishimori se referă la tranziția dintre fazele cu ordine pe distanță scurtă și lungă în modelul Ising cu legături aleatoare. Pe un calculator cuantic, faza cu ordine pe distanță lungă se manifestă ca o stare în care qubiții sunt imbricați pe întregul dispozitiv. Această stare puternic imbricată este pregătită folosind protocolul *generarea imbricării prin măsurătoare* (GEM). Prin utilizarea măsurătorilor în mijlocul circuitului, protocolul GEM poate imbrica qubiții pe întregul dispozitiv folosind circuite de adâncime constantă. Acest tutorial folosește implementarea protocolului GEM din pachetul software [GEM Suite](https://github.com/qiskit-community/gem-suite).
## Cerințe
Înainte de a începe acest tutorial, asigură-te că ai instalate următoarele:

- Qiskit SDK v1.0 sau mai recent, cu suport pentru [vizualizare](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 sau mai recent (`pip install qiskit-ibm-runtime`)
- Qiskit Aer v0.14 sau mai recent (`pip install qiskit-aer`)
- GEM Suite (`pip install gem-suite`)
## Configurare

In [1]:
import matplotlib.pyplot as plt
import warnings

from collections import defaultdict

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator

from qiskit.transpiler import generate_preset_pass_manager

from gem_suite import PlaquetteLattice
from gem_suite.experiments import GemExperiment

## Exemplu la scară mică cu simulator

În această secțiune, fluxul de lucru complet este parcurs pe `AerSimulator` fără zgomot. Rețeaua de plachete este restricționată la o singură plachetă (12 qubiți), astfel încât simularea rămâne mică și rapidă, exercitând totuși fiecare parte a protocolului GEM: măsurători în mijlocul circuitului, parcurgerea unghiurilor $R_{ZZ}$, decodarea și analiza varianței normalizate. Același flux de lucru este scalat ulterior la mai multe plachete și la rețeaua completă pe hardware real.

### Pasul 1: Maparea intrărilor clasice la o problemă cuantică

Protocolul GEM funcționează pe un procesor cuantic cu conectivitate a qubiților descrisă de o rețea. Procesoarele cuantice IBM Quantum&reg; actuale folosesc [rețeaua hex grea](https://www.ibm.com/quantum/blog/heavy-hex-lattice). Qubiții procesorului sunt grupați în *plachete* în funcție de celula unitară a rețelei pe care o ocupă. Deoarece un qubit poate apărea în mai mult de o celulă unitară, plachetele nu sunt disjuncte. Pe rețeaua hex grea, o plachetă conține 12 qubiți. Plachetele înseși formează și ele o rețea, în care două plachete sunt conectate dacă împart qubiți. Pe rețeaua hex grea, plachetele vecine împart trei qubiți.

În pachetul software GEM Suite, clasa fundamentală pentru implementarea protocolului GEM este `PlaquetteLattice`, care reprezintă rețeaua de plachete (distinctă de rețeaua hex grea). Un `PlaquetteLattice` poate fi inițializat dintr-o hartă de cuplare a qubiților. În prezent, sunt suportate doar hărțile de cuplare hex grele.

Următoarea celulă de cod inițializează o rețea de plachete din harta de cuplare a unui procesor cuantic (QPU). Rețeaua de plachete nu acoperă întotdeauna întregul hardware. De exemplu, `ibm_torino` are 133 de qubiți în total, dar cea mai mare rețea de plachete care se potrivește pe dispozitiv folosește doar 125 dintre ei, cuprinzând 18 plachete; `ibm_pittsburgh` (156 qubiți) încadrează similar 144 de qubiți în 21 de plachete. Același tipar se aplică și pentru alte QPU-uri hex grele cu număruri diferite de qubiți.

In [ ]:
# QiskitRuntimeService.save_account(channel="ibm_quantum", token="<YOUR_API_KEY>", overwrite=True,
# set_as_default=True)
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
aer_backend = AerSimulator.from_backend(backend)
plaquette_lattice = PlaquetteLattice.from_coupling_map(backend.coupling_map)

print(f"Number of qubits in backend: {backend.num_qubits}")
print(
    f"Number of qubits in plaquette lattice: {len(list(plaquette_lattice.qubits()))}"
)
print(f"Number of plaquettes: {len(list(plaquette_lattice.plaquettes()))}")

Poți vizualiza rețeaua de plachete generând o diagramă a reprezentării sale grafice. În diagramă, plachetele sunt reprezentate de hexagoane etichetate, iar două plachete sunt conectate printr-o muchie dacă împart qubiți.

In [3]:
plaquette_lattice.draw_plaquettes()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif)

Poți obține informații despre plachete individuale, cum ar fi qubiții pe care îi conțin, folosind metoda `plaquettes`.

In [4]:
# Get a list of the plaquettes
plaquettes = list(plaquette_lattice.plaquettes())
# Display information about plaquette 0
plaquettes[0]

PyPlaquette(index=0, qubits=[3, 4, 5, 6, 7, 16, 17, 23, 24, 25, 26, 27], neighbors=[4, 3, 1])

You can also produce a diagram of the underlying qubits that form the plaquette lattice.

In [5]:
plaquette_lattice.draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif" alt="Output of the previous code cell" />

Poți produce și o diagramă a qubiților de bază care formează rețeaua de plachete.

In [6]:
# Filter the plaquette lattice down to a single plaquette (12 qubits)
# so the AerSimulator run stays fast. The full lattice is used later
# in the large-scale hardware example.
gem_exp = GemExperiment(plaquette_lattice.filter([9]), backend=aer_backend)

# visualize the plaquette lattice after filtering
plaquette_lattice.filter([9]).draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif)

Pe lângă etichetele qubiților și muchiile care indică qubiții conectați, diagrama conține trei informații suplimentare relevante pentru protocolul GEM:
- Fiecare qubit este fie umbrit (gri), fie neumbrit. Qubiții umbriți sunt qubiți de „sit" care reprezintă siturile modelului Ising, iar qubiții neumbriți sunt qubiți de „legătură" folosiți pentru a media interacțiunile dintre qubiții de sit.
- Fiecare qubit de sit este etichetat fie (A), fie (B), indicând unul dintre cele două roluri pe care un qubit de sit le poate juca în protocolul GEM (rolurile sunt explicate mai târziu).
- Fiecare muchie este colorată cu una din șase culori, împărțind astfel muchiile în șase grupe. Această împărțire determină modul în care porțile cu doi qubiți pot fi paralelizate, precum și diferite tipare de planificare care pot produce cantități diferite de erori pe un procesor cuantic zgomotos. Deoarece muchiile dintr-o grupă sunt disjuncte, un strat de porți cu doi qubiți poate fi aplicat simultan pe acele muchii. De fapt, este posibil să se împartă cele șase culori în trei grupe de câte două culori astfel încât reuniunea fiecărei grupe de două culori să fie în continuare disjunctă. Prin urmare, sunt necesare doar trei straturi de porți cu doi qubiți pentru a activa fiecare muchie. Există 12 moduri de a împărți astfel cele șase culori, iar fiecare astfel de împărțire produce un program diferit de porți cu 3 straturi.

Acum că ai creat o rețea de plachete, pasul următor este să inițializezi un obiect `GemExperiment`, transmițând atât rețeaua de plachete, cât și Backend-ul pe care intenționezi să rulezi experimentul. Clasa `GemExperiment` gestionează implementarea efectivă a protocolului GEM, inclusiv generarea circuitelor, trimiterea joburilor și analiza datelor. Următoarea celulă de cod inițializează clasa de experiment restricționând totodată rețeaua de plachete la o singură plachetă (12 qubiți), menținând simularea mică și rapidă. Rețeaua completă de plachete este folosită mai târziu la scalarea spre hardware real.

In [7]:
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

Total number of circuits: 252


![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif)

Un Circuit al protocolului GEM este construit urmând pașii de mai jos:
1. Pregătește starea $|+\rangle$ totală aplicând o poartă Hadamard fiecărui qubit.
2. Aplică o poartă $R_{ZZ}$ între fiecare pereche de qubiți conectați. Aceasta poate fi realizată folosind trei straturi de porți. Fiecare poartă $R_{ZZ}$ acționează pe un qubit de sit și un qubit de legătură. Dacă qubit-ul de sit este etichetat (B), atunci unghiul este fixat la $\frac{\pi}{2}$. Dacă qubit-ul de sit este etichetat (A), atunci unghiul poate varia, producând circuite diferite. În mod implicit, intervalul unghiurilor este setat la 21 de puncte egal distanțate între $0$ și $\frac{\pi}{2}$, inclusiv.
3. Măsoară fiecare qubit de legătură în baza Pauli $X$. Deoarece qubiții sunt măsurați în baza Pauli $Z$, aceasta poate fi realizată aplicând o poartă Hadamard înainte de a măsura qubit-ul.

Rețineți că articolul citat în introducerea acestui tutorial folosește o convenție diferită pentru unghiul $R_{ZZ}$, care diferă de convenția folosită în acest tutorial printr-un factor de 2.

La pasul 3, sunt măsurați doar qubiții de legătură. Pentru a înțelege în ce stare rămân qubiții de sit, este instructiv să considerăm cazul în care unghiul $R_{ZZ}$ aplicat qubiților de sit (A) la pasul 2 este egal cu $\frac{\pi}{2}$. În acest caz, qubiții de sit rămân într-o stare puternic imbricată similară stării GHZ,

$$
\lvert \text{GHZ} \rangle = \lvert 00 \cdots 00 \rangle + \lvert 11 \cdots 11 \rangle.
$$

Din cauza aleatoriei în rezultatele măsurătorilor, starea reală a qubiților de sit ar putea fi o stare diferită cu ordine pe distanță lungă, de exemplu, $\lvert 00110 \rangle + \lvert 11001 \rangle$. Cu toate acestea, starea GHZ poate fi recuperată prin aplicarea unei operații de decodare bazate pe rezultatele măsurătorilor. Când unghiul $R_{ZZ}$ este redus față de $\frac{\pi}{2}$, ordinea pe distanță lungă poate fi în continuare recuperată până la un unghi critic, care în absența zgomotului este aproximativ $0.3 \pi$. Sub acest unghi, starea rezultată nu mai prezintă imbricare pe distanță lungă. Această tranziție dintre prezența și absența ordinii pe distanță lungă este tranziția de fază Nishimori.

În descrierea de mai sus, qubiții de sit au rămas nemăsurați, iar operația de decodare poate fi efectuată aplicând porți cuantice. În experimentul implementat în GEM Suite, pe care îl urmează acest tutorial, qubiții de sit sunt de fapt măsurați, iar operația de decodare este aplicată într-un pas de post-procesare clasică.

În descrierea de mai sus, operația de decodare poate fi efectuată aplicând porți cuantice qubiților de sit pentru a recupera starea cuantică. Cu toate acestea, dacă scopul este de a măsura imediat starea (de exemplu, în scopuri de caracterizare), atunci qubiții de sit sunt măsurați împreună cu qubiții de legătură, iar operația de decodare poate fi aplicată într-un pas de post-procesare clasică.

Pe lângă dependența de unghiul $R_{ZZ}$ de la pasul 2, care în mod implicit parcurge 21 de valori, Circuitul protocolului GEM depinde și de tiparul de planificare folosit pentru a implementa cele trei straturi de porți $R_{ZZ}$. Așa cum s-a discutat anterior, există 12 astfel de tipare de planificare. Prin urmare, numărul total de circuite din experiment este $21 \times 12 = 252$.

Circuitele experimentului pot fi generate folosind metoda `circuits` a clasei `GemExperiment`.

In [8]:
# Restrict experiment to the first scheduling pattern
gem_exp.set_experiment_options(schedule_idx=0)

# There are less circuits now
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

# Print the RZZ angles swept over
print(f"RZZ angles:\n{gem_exp.parameters()}")

Total number of circuits: 21
RZZ angles:
[0.         0.07853982 0.15707963 0.23561945 0.31415927 0.39269908
 0.4712389  0.54977871 0.62831853 0.70685835 0.78539816 0.86393798
 0.9424778  1.02101761 1.09955743 1.17809725 1.25663706 1.33517688
 1.41371669 1.49225651 1.57079633]


The following code cell draws a diagram of the circuit at index 5. To reduce the size of the diagram, the measurement gates at the end of the circuit are removed.

In [9]:
# Get the circuit at index 5
circuit = circuits[5]
# Remove the final measurements to ease visualization
circuit.remove_final_measurements()
# Draw the circuit
circuit.draw("mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif" alt="Output of the previous code cell" />

În scopul acestui tutorial, este suficient să considerăm un singur tipar de planificare. Următoarea celulă de cod restricționează experimentul la primul tipar de planificare. Ca urmare, experimentul are doar 21 de circuite, câte unul pentru fiecare unghi $R_{ZZ}$ parcurs.

In [10]:
# Demonstrate setting transpile options
gem_exp.set_transpile_options(
    optimization_level=1  # This is the default optimization level
)
pass_manager = generate_preset_pass_manager(
    backend=aer_backend,
    initial_layout=list(gem_exp.physical_qubits),
    **dict(gem_exp.transpile_options),
)
transpiled = pass_manager.run(circuit)
transpiled.draw("mpl", idle_wires=False, fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

To execute the GEM protocol circuits on the hardware, call the `run` method of the `GemExperiment` object. You can specify the number of shots you want to sample from each circuit. The `run` method returns an [ExperimentData](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.framework.ExperimentData.html) object which you should save to a variable. Note that the `run` method only submits jobs without waiting for them to finish, so it is a non-blocking call.

In [11]:
exp_data = gem_exp.run(shots=10_000)

Următoarea celulă de cod desenează o diagramă a circuitului de la indexul 5. Pentru a reduce dimensiunea diagramei, porțile de măsurare de la sfârșitul circuitului sunt eliminate.

In [12]:
# The noiseless AerSimulator produces zero-variance UFloat objects in the
# analysis, which triggers a harmless warning from the `uncertainties`
# library. Suppress it so the output stays clean.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", message="Using UFloat objects with std_dev==0"
    )
    exp_data.block_for_results()
exp_data

ExperimentData(GemExperiment, 90bf2a90-f729-4c4e-a6da-664aecb11039, job_ids=['04a7c405-47fd-46ca-aa4b-aaf7e339cfbe'], metadata=<5 items>, figure_names=['two_point_correlation.svg', 'normalized_variance.svg', 'plaquette_ops.svg', 'bond_ops.svg'])

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif)

### Pasul 2: Optimizarea problemei pentru execuția pe hardware cuantic
Transpilarea circuitelor cuantice pentru execuție pe hardware implică de obicei un [număr de etape](/guides/transpiler-stages). De regulă, etapele care implică cel mai mare efort de calcul sunt alegerea layout-ului de qubiți, rutarea porților cu doi qubiți pentru a respecta conectivitatea qubiților din hardware și optimizarea circuitului pentru a minimiza numărul de porți și adâncimea. În protocolul GEM, etapele de layout și rutare sunt inutile deoarece conectivitatea hardware este deja incorporată în design-ul protocolului. Circuitele au deja un layout de qubiți, iar porțile cu doi qubiți sunt deja mapate pe conexiunile native. Mai mult, pentru a păstra structura circuitului pe măsură ce unghiul $R_{ZZ}$ variază, ar trebui efectuată doar o optimizare de circuit foarte de bază.

Clasa `GemExperiment` transpilează transparent circuitele la executarea experimentului. Etapele de layout și rutare sunt deja suprascrise implicit să nu facă nimic, iar optimizarea circuitului este efectuată la un nivel care optimizează doar porțile cu un singur qubit. Cu toate acestea, poți suprascrie sau transmite opțiuni suplimentare folosind metoda `set_transpile_options`. De dragul vizualizării, următoarea celulă de cod transpilează manual circuitul afișat anterior și desenează circuitul transpilat.

In [13]:
def magnetization_distribution(
    counts_dict: dict[str, int],
) -> dict[str, float]:
    """Compute magnetization distribution from counts dictionary."""
    # Construct dictionary from magnetization to count
    mag_dist = defaultdict(float)
    for bitstring, count in counts_dict.items():
        mag = bitstring.count("0") - bitstring.count("1")
        mag_dist[mag] += count
    # Normalize
    shots = sum(counts_dict.values())
    for mag in mag_dist:
        mag_dist[mag] /= shots
    return mag_dist


# Get counts dictionaries with and without decoding
data = exp_data.data()
# Get the last data point, which is at the angle for the GHZ state
raw_counts = data[-1]["counts"]
# Without decoding
site_indices = [
    i for i, q in enumerate(gem_exp.plaquettes.qubits()) if q.role == "Site"
]
site_raw_counts = defaultdict(int)
for key, val in raw_counts.items():
    site_str = "".join(key[-1 - i] for i in site_indices)
    site_raw_counts[site_str] += val
# With decoding
_, site_decoded_counts = gem_exp.plaquettes.decode_outcomes(
    raw_counts, return_counts=True
)

# Compute magnetization distribution
raw_magnetization = magnetization_distribution(site_raw_counts)
decoded_magnetization = magnetization_distribution(site_decoded_counts)

# Plot
plt.bar(*zip(*raw_magnetization.items()), label="raw")
plt.bar(*zip(*decoded_magnetization.items()), label="decoded", width=0.3)
plt.legend()
plt.xlabel("Magnetization")
plt.ylabel("Frequency")
plt.title("Magnetization distribution with and without decoding")

Text(0.5, 1.0, 'Magnetization distribution with and without decoding')

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/8ead3582-16df-4616-836c-bdce867ad6b8-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif)

### Pasul 3: Executarea folosind primitivele Qiskit
Pentru a executa circuitele protocolului GEM pe hardware, apelează metoda `run` a obiectului `GemExperiment`. Poți specifica numărul de shot-uri pe care vrei să le eșantionezi din fiecare Circuit. Metoda `run` returnează un obiect [ExperimentData](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.framework.ExperimentData.html) pe care ar trebui să îl salvezi într-o variabilă. Rețineți că metoda `run` doar trimite joburi fără a aștepta finalizarea lor, deci este un apel neblocant.

In [14]:
exp_data.figure("two_point_correlation")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/4ecb25c8-e572-49af-a879-9943039db131-0.avif" alt="Output of the previous code cell" />

Pentru a aștepta rezultatele, apelează metoda `block_for_results` a obiectului `ExperimentData`. Acest apel va face ca interpretorul să aștepte până când joburile sunt finalizate.

In [15]:
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/2b351d68-3924-445a-94ef-047b16214e8a-0.avif" alt="Output of the previous code cell" />

## Large-scale hardware example

Having validated the protocol on a simulator, you can now scale up the experiment and run it on the real quantum hardware backend selected in the [Setup](#setup) section. This example uses two larger problem sizes:

- **Six plaquettes (~49 qubits)**: a mid-size run that already shows the rightward shift of the critical point under hardware noise.
- **The full plaquette lattice**: every plaquette the device's heavy-hex topology supports (for example, 18 plaquettes / 125 qubits on `ibm_torino` or 21 plaquettes / 144 qubits on `ibm_pittsburgh`), entangling qubits across the entire device with constant-depth circuits.

The single code cell below is self-contained: it builds the plaquette lattice from the backend's coupling map and runs both experiments, so this section can be executed after the [Setup](#setup) cells without first running the small-scale section.

In [ ]:
# -------------------------Step 1-------------------------
# Initialize the runtime service, pick a real quantum hardware backend,
# and build the plaquette lattice from its coupling map. This is repeated
# from the small-scale example so this cell can run standalone after the
# Setup section. The full plaquette lattice is the "large-scale" target;
# a six-plaquette subset (range(3, 9)) is also used to show an intermediate
# scaling step.
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
plaquette_lattice = PlaquetteLattice.from_coupling_map(backend.coupling_map)

# Build a GemExperiment for the full plaquette lattice and one for the
# six-plaquette subset, each restricted to a single scheduling pattern so
# the experiment has one circuit per RZZ angle (21 circuits total).
gem_exp_full = GemExperiment(plaquette_lattice, backend=backend)
gem_exp_full.set_experiment_options(schedule_idx=0)
gem_exp_6 = GemExperiment(
    plaquette_lattice.filter(range(3, 9)), backend=backend
)
gem_exp_6.set_experiment_options(schedule_idx=0)

circuits = gem_exp_full.circuits()
print(f"Total number of circuits (full lattice): {len(circuits)}")

# -------------------------Step 2-------------------------
# GemExperiment transpiles internally for the target backend: the layout
# and routing stages are overridden because the plaquette lattice already
# matches the hardware connectivity, and optimization is restricted so the
# RZZ angle structure is preserved. The code below manually transpiles one
# circuit from the six-plaquette experiment with the same settings this
# experiment will use, and draws it for inspection. (The full-lattice
# transpiled circuit has too many qubits to visualize cleanly, so the
# six-plaquette circuit is used here as a representative example.)
gem_exp_6.set_transpile_options(optimization_level=1)
circuits_6 = gem_exp_6.circuits()
pass_manager = generate_preset_pass_manager(
    backend=backend,
    initial_layout=list(gem_exp_6.physical_qubits),
    **dict(gem_exp_6.transpile_options),
)
transpiled = pass_manager.run(circuits_6[5])
display(transpiled.draw("mpl", idle_wires=False, fold=-1, scale=0.5))

# -------------------------Step 3-------------------------
# Run both problem sizes on real hardware:
#   1. Six plaquettes (~49 qubits) — an intermediate scale-up.
#   2. The full plaquette lattice — every plaquette the device supports.
exp_data_6 = gem_exp_6.run(shots=10_000, job_tags=["TUT_NPT"])
exp_data_full = gem_exp_full.run(shots=10_000, job_tags=["TUT_NPT"])
exp_data_6.block_for_results()
exp_data_full.block_for_results()

# -------------------------Step 4-------------------------
# Plot the normalized variance at each scale. The peak marks the critical
# point of the Nishimori transition; as the system grows, hardware noise
# shifts the peak rightward.
display(exp_data_6.figure("normalized_variance"))
exp_data_full.figure("normalized_variance")

Total number of circuits (full lattice): 21


<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-1.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-2.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-3.avif" alt="Output of the previous code cell" />

### Pasul 4: Post-procesarea și returnarea rezultatului în formatul clasic dorit
La un unghi $R_{ZZ}$ de $\frac{\pi}{2}$, starea decodată ar fi starea GHZ în absența zgomotului. Ordinea pe distanță lungă a stării GHZ poate fi vizualizată prin reprezentarea grafică a magnetizării șirurilor de biți măsurate. Magnetizarea $M$ este definită ca suma operatorilor Pauli $Z$ cu un singur qubit,
$$
M = \sum_{j=1}^N Z_j,
$$
unde $N$ este numărul de qubiți de sit. Valoarea sa pentru un șir de biți este egală cu diferența dintre numărul de zerouri și numărul de unuri. Măsurarea stării GHZ produce starea cu toți zeroii sau starea cu toți unii cu probabilitate egală, deci magnetizarea ar fi $+N$ jumătate din timp și $-N$ cealaltă jumătate. În prezența erorilor datorate zgomotului, ar apărea și alte valori, dar dacă zgomotul nu este prea mare, distribuția ar fi în continuare concentrată în apropierea $+N$ și $-N$.

Pentru șirurile de biți brute înainte de decodare, distribuția magnetizării ar fi echivalentă cu cea a șirurilor de biți uniform aleatoare, în absența zgomotului.

Următoarea celulă de cod reprezintă grafic magnetizarea șirurilor de biți brute și a celor decodate la unghiul $R_{ZZ}$ de $\frac{\pi}{2}$.